# Stepper Motor Test Notebook

Small test notebook for the L6470-based stepper controller, talking ASCII commands over a serial port (commands terminated by `\n`, see `stepper motor instruction set.pdf`).

**Before running:**
1. Install pyserial: `pip install pyserial`
2. Set `PORT` below to the right COM port (Windows: `COM3`, `COM4`, ... / Linux: `/dev/ttyUSB0`).
3. Verify `BAUD` matches the controller (115200 is the common default; try 9600 / 19200 if no response).
4. Make sure nothing is in the path of the motor — start with small `rotate` values.

Each test cell is small and stoppable. Run them top-to-bottom; the last cell switches off the coils and closes the port.

In [6]:
import time
import serial
from serial.tools import list_ports

# Quick helper: list available serial ports so you can pick the right COM number.
for p in list_ports.comports():
    print(p.device, '-', p.description)

COM5 - Arduino Uno (COM5)


In [10]:
PORT = 'COM5'        # <-- change to your port
BAUD = 9600        # try 9600 / 19200 / 38400 / 115200 if the controller does not respond
MOTOR = 1            # motor number (the [n] in the command list)
READ_TIMEOUT = 1.0   # seconds

In [41]:
class Stepper:
    """Thin wrapper around the ASCII command set from the L6470 controller."""

    def __init__(self, port, baud=115200, timeout=1.0, motor=1):
        self.ser = serial.Serial(port, baud, timeout=timeout)
        self.motor = motor
        time.sleep(2.0)  # most USB-serial bridges reset on open; give the controller time
        self.ser.reset_input_buffer()

    def close(self):
        if self.ser.is_open:
            self.ser.close()

    def _send(self, cmd):
        line = (cmd + '\n').encode('ascii')
        self.ser.write(line)
        self.ser.flush()

    def _read_reply(self):
        # Read every line that arrives until the timeout expires.
        lines = []
        deadline = time.time() + self.ser.timeout
        while time.time() < deadline:
            raw = self.ser.readline()
            if not raw:
                break
            lines.append(raw.decode('ascii', errors='replace').strip())
        return lines

    def cmd(self, cmd, expect_reply=False):
        """Send a raw command string. Returns the reply lines (possibly empty)."""
        self._send(cmd)
        if expect_reply:
            return self._read_reply()
        time.sleep(0.05)
        return self._read_reply()  # drain anything the controller echoed

    # --- Commands from the instruction set --------------------------------
    def reset(self):                       return self.cmd('reset')
    def gotoswitch(self):                  return self.cmd(f'gotoswitch {self.motor}')
    def busy(self):                        return self.cmd(f'busy {self.motor}', expect_reply=True)
    def wait(self):                        return self.cmd(f'wait {self.motor}')
    def rotate(self, steps):               return self.cmd(f'rotate {self.motor} {int(steps)}')
    def setprofile(self, microsteps=8, acceleration=1000, speed=500):
        return self.cmd(f'setprofile {self.motor} {int(microsteps)} {int(acceleration)} {int(speed)}')
    def setvoltage(self, driving=6.8, holding=2.0):
        return self.cmd(f'setvoltage {self.motor} {driving} {holding}')
    def getvoltage(self):                  return self.cmd('getvoltage', expect_reply=True)
    def setpos(self, step):                return self.cmd(f'setpos {self.motor} {int(step)}')
    def getpos(self):                      return self.cmd(f'getpos {self.motor}', expect_reply=True)
    def clearpos(self):                    return self.cmd(f'clearpos {self.motor}')
    def run(self, speed):                  return self.cmd(f'run {self.motor} {int(speed)}')
    def softstop(self):                    return self.cmd(f'softstop {self.motor}')
    def hardstop(self):                    return self.cmd(f'hardstop {self.motor}')
    def hiz(self):                         return self.cmd(f'hiz {self.motor}')

## 1. Open the port and check the link

In [42]:
try: mot.close()
except Exception as e: print(e)
mot = Stepper(PORT, BAUD, timeout=READ_TIMEOUT, motor=MOTOR)
print('Port open:', mot.ser.is_open, '->', mot.ser.port, '@', mot.ser.baudrate)
print('Supply voltage reply:', mot.getvoltage())

Port open: True -> COM5 @ 9600
Supply voltage reply: ['11.9']


## 2. Reset and configure profile + voltages
Using the PDF defaults (microsteps=8, accel=1000, speed=500, driving=2.8 V, holding=1.0 V). The PDF has handwritten 6.8 V driving / 2 V holding annotations — only switch to those if you know your motor and supply can take it.

In [43]:
mot.reset()
mot.setprofile(microsteps=8, acceleration=1000, speed=500)
mot.setvoltage(driving=6.8, holding=2.0)
mot.clearpos()
print('Position after clearpos:', mot.getpos())

Position after clearpos: ['0']


## 3. Rotate a small amount and read back position
With microsteps=8, a full revolution (400-step motor) = 400 * 8 = 32000 steps. Rotate 400 steps (~quarter turn) first as a sanity check — small enough that nothing dramatic happens if direction or wiring is off.

In [44]:
mot.rotate(3200)
mot.wait()
print('After +3200 steps:', mot.getpos())

After +3200 steps: ['3200']


In [45]:
mot.rotate(-3200)
mot.wait()
print('After -3200 steps (should be ~0):', mot.getpos())

After -3200 steps (should be ~0): ['0']


## 4. Endless run + softstop
Run at 300 steps/s for 2 seconds, then decelerate to stop. If anything looks wrong, interrupt the kernel and run the 'switch off' cell at the bottom.

In [46]:
mot.run(300)
time.sleep(2.0)
mot.softstop()
mot.wait()
print('Position after run+softstop:', mot.getpos())

Position after run+softstop: ['113']


## 5. Switch off coils and close port
Always finish with `hiz` so the holding current is removed and the motor isn't left warm.

In [26]:
mot.hiz()
mot.close()
print('Port closed:', not mot.ser.is_open)

Port closed: True


### Emergency stop
If something goes wrong, run this cell:

In [ ]:
try:
    mot.hardstop()
    mot.hiz()
finally:
    mot.close()